# 排序数据集构建 - Data Process

从 rating、movie、tag、genome 等原始表构建用于排序/推荐模型训练的宽表，包含：
- Label：打分 >= 4.0 视为喜欢 (1)，否则为不喜欢 (0)
- Item 特征：电影流派、热度、平均分、Genome 标签
- User 特征：用户评分习惯、常用 Tag
- Context：时间（小时、星期几）

## 1. 导入与数据读取

In [2]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("rating.csv")
movies = pd.read_csv("movie.csv")
tags = pd.read_csv("tag.csv")
genome_scores = pd.read_csv("genome_scores.csv")
genome_tags = pd.read_csv("genome_tags.csv")
print("ratings.head():")
print(ratings.head(5))
print("\nmovies.head():")
print(movies.head(5))
print("\ntags.head():")
print(tags.head(5))
print("\ngenome_scores.head():")
print(genome_scores.head(5))
print("\ngenome_tags.head():")
print(genome_tags.head(5))

ratings.head():
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40

movies.head():
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

tags.head():
   userId  movieId            tag            timestamp
0      18     41

## 2. 确定 Label (Target)

以 rating.csv 为主表，打分 >= 4.0 视为喜欢 (Label=1)，否则为不喜欢 (Label=0)。

In [3]:
dataset = ratings[["userId", "movieId", "rating", "timestamp"]].copy()
dataset["label"] = (dataset["rating"] >= 4.0).astype(int)
print(dataset)

          userId  movieId  rating            timestamp  label
0              1        2     3.5  2005-04-02 23:53:47      0
1              1       29     3.5  2005-04-02 23:31:16      0
2              1       32     3.5  2005-04-02 23:33:39      0
3              1       47     3.5  2005-04-02 23:32:07      0
4              1       50     3.5  2005-04-02 23:29:40      0
...          ...      ...     ...                  ...    ...
20000258  138493    68954     4.5  2009-11-13 15:42:00      1
20000259  138493    69526     4.5  2009-12-03 18:31:48      1
20000260  138493    69644     3.0  2009-12-07 18:10:57      0
20000261  138493    70286     5.0  2009-11-13 15:42:24      1
20000262  138493    71619     2.5  2009-10-17 20:25:36      0

[20000263 rows x 5 columns]


## 3. 构建 Item Features (电影特征)

- 3.1 电影流派 (Genres) 转化为列表
- 3.2 电影热度和平均得分
- 3.3 电影的高频 Genome 语义标签（每部电影 relevance 最高的前 5 个 tagId）

In [4]:
# 3.1 电影流派转化为列表
movies["genres_list"] = movies["genres"].apply(lambda x: x.split("|"))
print("3.1 电影流派转化为列表后的 movies.head():")
print(movies[["movieId", "genres", "genres_list"]].head())

# 3.2 电影热度和平均得分
item_stats = (
    ratings.groupby("movieId")
    .agg(item_rating_count=("rating", "count"), item_rating_mean=("rating", "mean"))
    .reset_index()
)
print("\n3.2 每部电影的热度和平均得分 item_stats.head():")
print(item_stats.head())

# 3.3 电影的高频 Genome 语义标签：每部电影 relevance 最高的前 5 个 tagId
top_genomes = genome_scores.sort_values(
    ["movieId", "relevance"], ascending=[True, False]
)
top_genomes = top_genomes.groupby("movieId").head(5)
print("\n3.3 top_genomes（每部电影前5个relevance最高的genome tag）.head(10):")
print(top_genomes.head(10))

item_genome_tags = (
    top_genomes.groupby("movieId")["tagId"]
    .apply(list)
    .reset_index(name="item_top_genome_tags")
)
print("\n3.3 item_genome_tags.head():")
print(item_genome_tags.head())

# 合并 Item 特征
item_features = movies[["movieId", "genres_list"]].merge(
    item_stats, on="movieId", how="left"
)
item_features = item_features.merge(item_genome_tags, on="movieId", how="left")
print("\n最终 item_features.head():")
print(item_features.head())

3.1 电影流派转化为列表后的 movies.head():
   movieId                                       genres  \
0        1  Adventure|Animation|Children|Comedy|Fantasy   
1        2                   Adventure|Children|Fantasy   
2        3                               Comedy|Romance   
3        4                         Comedy|Drama|Romance   
4        5                                       Comedy   

                                         genres_list  
0  [Adventure, Animation, Children, Comedy, Fantasy]  
1                     [Adventure, Children, Fantasy]  
2                                  [Comedy, Romance]  
3                           [Comedy, Drama, Romance]  
4                                           [Comedy]  

3.2 每部电影的热度和平均得分 item_stats.head():
   movieId  item_rating_count  item_rating_mean
0        1              49695          3.921240
1        2              22243          3.211977
2        3              12735          3.151040
3        4               2756          2.861393
4      

## 4. 构建 User Features (用户特征)

- 4.1 用户的打分习惯（评分次数、平均分）
- 4.2 用户的标签偏好：从 tag.csv 统计用户打过最多次数的前 3 个 Tag

In [5]:
# 4.1 用户的打分习惯
user_stats = (
    ratings.groupby("userId")
    .agg(user_rating_count=("rating", "count"), user_rating_mean=("rating", "mean"))
    .reset_index()
)
print("4.1 用户的打分习惯 user_stats.head():")
print(user_stats.head())

# 4.2 用户的标签偏好：统计用户打过最多次数的前 3 个 Tag
user_tags = (
    tags.groupby("userId")["tag"]
    .apply(lambda x: list(x.value_counts().index[:3]))
    .reset_index(name="user_top_tags")
)
print("\n4.2 用户的标签偏好 user_tags.head():")
print(user_tags.head())

# 合并 User 特征
user_features = user_stats.merge(user_tags, on="userId", how="left")
print("\n合并后的 user_features.head():")
print(user_features.head())

4.1 用户的打分习惯 user_stats.head():
   userId  user_rating_count  user_rating_mean
0       1                175          3.742857
1       2                 61          4.000000
2       3                187          4.122995
3       4                 28          3.571429
4       5                 66          4.272727

4.2 用户的标签偏好 user_tags.head():
   userId                             user_top_tags
0      18                             [Mark Waters]
1      65         [noir thriller, jesus, dark hero]
2      96        [animation, beautiful, characters]
3     121         [Nudity (Topless), drugs, comedy]
4     129  [Adam Sandler, high school, Ben Stiller]

合并后的 user_features.head():
   userId  user_rating_count  user_rating_mean user_top_tags
0       1                175          3.742857           NaN
1       2                 61          4.000000           NaN
2       3                187          4.122995           NaN
3       4                 28          3.571429           NaN
4       5  

## 5. 组装最终的大宽表 (Join Master Table)

将主表与用户/物品特征表拼接，提取时间特征，剔除不参与训练的列，并做缺失值填充。

In [6]:
# 主表与特征表拼接
dataset = dataset.merge(user_features, on="userId", how="left")
dataset = dataset.merge(item_features, on="movieId", how="left")

# 提取 Context 时间特征
dataset["timestamp"] = pd.to_datetime(dataset["timestamp"])
dataset["hour"] = dataset["timestamp"].dt.hour
dataset["dayofweek"] = dataset["timestamp"].dt.dayofweek

# 剔除不参与训练的列
final_ranking_dataset = dataset.drop(columns=["rating", "timestamp"])

# 缺失值填充：没有打过 tag 的用户 / 没有 genome 的电影用空列表填充
final_ranking_dataset["user_top_tags"] = final_ranking_dataset["user_top_tags"].apply(
    lambda x: x if isinstance(x, list) else []
)
final_ranking_dataset["item_top_genome_tags"] = final_ranking_dataset[
    "item_top_genome_tags"
].apply(lambda x: x if isinstance(x, list) else [])

## 6. 查看结果

In [7]:
final_ranking_dataset.head()
# final_ranking_dataset.info()  # 可选：查看列类型与缺失情况
# 保留前 k 个数据，这里以 xk=30000 为示例
k = 300000
final_ranking_dataset_head = final_ranking_dataset.head(k)
final_ranking_dataset_head.to_csv(f"final_ranking_dataset_{k//1000}k.csv", index=False)
print(f"final_ranking_dataset 已保存到 final_ranking_dataset_{k//1000}k.csv")

final_ranking_dataset 已保存到 final_ranking_dataset_300k.csv


In [8]:
# 统计final_ranking_dataset_head的userid数量
num_users = final_ranking_dataset_head["userId"].nunique()
print(f"在前{k}条数据中，共有用户数: {num_users}")

在前300000条数据中，共有用户数: 2054
